# 打ち間違いを許してローマ字をかなに変換する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_japanese_input/romaji_fuzzy.ipynb)

NLP クックブック 28-02 ／ 日本語入力（かな漢字変換）

ローマ字入力で、キーの打ち間違いや文字の抜けがあっても、意図した単語を推定して変換したい。日本語を学び始めた人の、つづりの誤りにも対応したい。

In [1]:
!pip install -q jaconv fugashi unidic-lite

In [2]:
import heapq, io, re, urllib.request, zipfile
from collections import Counter
import jaconv
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text)

# 単語の読み (かな) をローマ字にした辞書を作る: ローマ字 → {(かな, 表記): 頻度}
tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
words = Counter()
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for w in tagger(aozora(base + f)):
        if w.feature.kana and w.feature.pos1 in ("名詞", "動詞", "形容詞", "副詞", "形状詞") and re.fullmatch(r"[ァ-ヶー]+", w.feature.kana):
            words[jaconv.kata2hira(w.feature.kana), w.surface] += 1
lexicon = {}
for (kana, surface), n in words.items():
    if n >= 3 and len(kana) >= 2:
        lexicon.setdefault(jaconv.kana2alphabet(kana), Counter())[kana, surface] += n
print(len(lexicon), "種類のローマ字つづり", list(lexicon.items())[:3])

5143 種類のローマ字つづり [('ichi', Counter({('いち', '一'): 716, ('いち', '位置'): 8, ('いち', '位地'): 4})), ('neko', Counter({('ねこ', '猫'): 240})), ('aru', Counter({('ある', 'ある'): 2208, ('ある', '有る'): 7}))]


In [3]:
QWERTY = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
NEIGHBORS = {c: set() for row in QWERTY for c in row}
for r, row in enumerate(QWERTY):
    for i, c in enumerate(row):
        NEIGHBORS[c] |= {row[j] for j in (i - 1, i + 1) if 0 <= j < len(row)}
        for other in (r - 1, r + 1):
            if 0 <= other < len(QWERTY):
                NEIGHBORS[c] |= {QWERTY[other][j] for j in (i - 1, i, i + 1) if 0 <= j < len(QWERTY[other])}

def cost_substitute(a, b):
    """置換のコスト: キーボードで隣のキーなら安く、母音どうしの取り違えも少し安くする"""
    if a == b:
        return 0.0
    if b in NEIGHBORS.get(a, ()):
        return 0.5
    return 0.8 if a in "aiueo" and b in "aiueo" else 1.0

def weighted_distance(typed, target, limit=2.5):
    """重み付き編集距離 (挿入・削除は 1、隣接文字の入れ替えは 0.5)。limit を超えたら打ち切る"""
    prev2, prev = None, [j * 1.0 for j in range(len(target) + 1)]
    for i in range(1, len(typed) + 1):
        cur = [i * 1.0]
        for j in range(1, len(target) + 1):
            best = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost_substitute(typed[i - 1], target[j - 1]))
            if i > 1 and j > 1 and typed[i - 1] == target[j - 2] and typed[i - 2] == target[j - 1]:
                best = min(best, prev2[j - 2] + 0.5)
            cur.append(best)
        if min(cur) > limit:
            return float("inf")
        prev2, prev = prev, cur
    return prev[-1]

print(weighted_distance("gakkou", "gakkou"), weighted_distance("gakkpu", "gakkou"), weighted_distance("gakkuo", "gakkou"), weighted_distance("gakou", "gakkou"), weighted_distance("gakkau", "gakkou"))

0.0 0.5 0.5 1.0 0.8


In [4]:
import math

def convert(typed, k=5):
    """距離が小さく、頻度が高い語を上位にする (スコア = 距離 − 頻度の対数の重み付き)"""
    scored = []
    for romaji, entries in lexicon.items():
        if abs(len(romaji) - len(typed)) <= 2:
            d = weighted_distance(typed, romaji)
            if d < float("inf"):
                for (kana, surface), n in entries.items():
                    scored.append((d - 0.15 * math.log(n), d, romaji, kana, surface))
    return [(surface, kana, romaji, d) for _, d, romaji, kana, surface in heapq.nsmallest(k, scored)]

for typed in ["gakkou", "gakkpu", "gakou", "sennsei", "sensie", "tomodati", "tomodachi", "benkyo", "kyuo", "arigatou", "muzukasii", "muzukashi"]:
    print(f"{typed:10} → {[(s, k) for s, k, _, d in convert(typed)][:4]}")

gakkou     → [('学校', 'がっこう'), ('退校', 'たいこう'), ('恰好', 'かっこう'), ('日光', 'にっこう')]
gakkpu     → [('学校', 'がっこう'), ('退校', 'たいこう'), ('恰好', 'かっこう'), ('厄介', 'やっかい')]
gakou      → [('学校', 'がっこう'), ('なく', 'なく'), ('額', 'がく'), ('かこう', 'かこう')]
sennsei    → [('先生', 'せんせい'), ('賛成', 'さんせい'), ('厭世', 'えんせい'), ('神聖', 'しんせい')]
sensie     → [('先生', 'せんせい'), ('戦争', 'せんそう'), ('賛成', 'さんせい'), ('厭世', 'えんせい')]
tomodati   → [('友達', 'ともだち'), ('ともかく', 'ともかく'), ('泊り', 'とまり'), ('留まり', 'とまり')]
tomodachi  → [('友達', 'ともだち'), ('飛び出し', 'とびだし'), ('取り出し', 'とりだし'), ('思い出し', 'おもいだし')]
benkyo     → [('勉強', 'べんきょう'), ('弁解', 'べんかい'), ('弁護', 'べんご'), ('便所', 'べんじょ')]
kyuo       → [('今日', 'きょう'), ('急', 'きゅう'), ('九', 'きゅう'), ('妙', 'みょう')]
arigatou   → [('ありがたい', 'ありがたい'), ('ありがたく', 'ありがたく'), ('ありがた', 'ありがた'), ('来月', 'らいげつ')]
muzukasii  → [('むずかしい', 'むずかしい'), ('恥ずかしい', 'はずかしい'), ('恥ずかし', 'はずかし'), ('むずかしく', 'むずかしく')]
muzukashi  → [('むずかしい', 'むずかしい'), ('恥ずかし', 'はずかし'), ('昔', 'むかし'), ('むずかしく', 'むずかしく')]


In [5]:
# 学習者に多いつづりの誤り (長音・促音・撥音の抜け) を、編集距離とは別に規則で候補にする
def learner_variants(typed):
    """長音の「u」や促音の子音の重なりが抜けた形から、補った候補を作る"""
    out = {typed}
    for i in range(len(typed)):
        if typed[i] in "ou":
            out.add(typed[:i + 1] + "u" + typed[i + 1:])                      # benkyo → benkyou
        if typed[i] in "kstp" and i > 0 and typed[i - 1] in "aiueo":
            out.add(typed[:i] + typed[i] + typed[i:])                         # gako → gakko
        if typed[i] == "n" and i + 1 < len(typed) and typed[i + 1] in "aiueoy":
            out.add(typed[:i] + "n" + typed[i:])                              # konichiwa → konnichiwa
    return out

def learner_variants_2(typed):
    """規則を 2 回まで重ねて適用する (gako → gakko → gakkou のように、誤りが 2 つある場合)"""
    return {v2 for v1 in learner_variants(typed) for v2 in learner_variants(v1)}

for typed in ["benkyo", "gako", "kino", "ryoko", "kekon", "sensei"]:
    hits = Counter()
    for v in learner_variants_2(typed):
        for (kana, surface), n in lexicon.get(v, {}).items():
            hits[surface, kana, v] += n
    print(typed, "→", hits.most_common(4))

benkyo → [(('勉強', 'べんきょう', 'benkyou'), 48)]
gako → [(('学校', 'がっこう', 'gakkou'), 238)]
kino → [(('昨日', 'きのう', 'kinou'), 32), (('きのう', 'きのう', 'kinou'), 6), (('機能', 'きのう', 'kinou'), 3)]
ryoko → [(('旅行', 'りょこう', 'ryokou'), 4)]
kekon → [(('結婚', 'けっこん', 'kekkon'), 46)]
sensei → [(('先生', 'せんせい', 'sensei'), 918)]


- 打ち間違いには傾向がある: 隣のキーを押す、2 文字の順序が入れ替わる、同じ文字を打ち損ねる・重ねる。操作ごとにコストを変えた編集距離を使うと、ただの編集距離より当たりやすくなる。
- 候補の順位は、「つづりの近さ」と「単語の出やすさ（頻度）」の両方で決める。雑音のある通信路モデルという考え方で、スペル訂正と同じ枠組みになる。
- 日本語学習者は、打ち間違いではなく、発音の聞き取りに由来する誤りをする（長音・促音・撥音の抜け、清音と濁音の取り違え）。キーボードの配置とは関係がないので、別の規則や重みを用意する。
- 辞書のすべての語と距離を計算するのは遅い。実用では、トライ木の上で編集距離を計算する、先頭の数文字で候補を絞る、SymSpell のような索引を使う、といった高速化をする。
- ここでは単語 1 つを対象にした。文全体では、かな漢字変換のラティスに「誤りを含む読み」の候補も加え、言語モデルでまとめて選ぶ。